# LoRA 가중치 HuggingFace 업로드

저장된 LoRA 가중치(`joseon_lora/`)를 HuggingFace Hub에 업로드하고,  
다시 다운로드해서 정상 동작하는지 확인합니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [ ]:
!pip install -q unsloth
!pip install -q huggingface_hub

## 2. HuggingFace 로그인

[https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 에서 **Write 권한** 토큰을 생성한 뒤, 아래 셀 실행 시 입력하세요.

In [ ]:
from huggingface_hub import login
login()

## 3. 저장된 LoRA 가중치 확인

`joseon_lora/` 폴더에 어떤 파일들이 저장되어 있는지 확인합니다.

In [ ]:
import os

LORA_DIR = "joseon_lora"

print(f"--- {LORA_DIR}/ 저장된 파일 ---")
for f in sorted(os.listdir(LORA_DIR)):
    size = os.path.getsize(os.path.join(LORA_DIR, f))
    if size > 1024 * 1024:
        print(f"  {f:40s} {size/1024/1024:.1f} MB")
    else:
        print(f"  {f:40s} {size/1024:.1f} KB")

## 4. HuggingFace Hub에 업로드

`HfApi.upload_folder()`로 `joseon_lora/` 폴더를 통째로 업로드합니다.

`REPO_ID`를 본인의 HuggingFace 사용자명으로 변경하세요.  
예: `"my-username/joseon-qwen2.5-1.5b-lora"`

### 모델 카드 작성

HuggingFace 레포지토리에 표시되는 모델 설명(README.md)을 작성합니다.  
`joseon_lora/` 폴더 안에 `README.md`를 만들어두면 업로드 시 자동으로 함께 올라갑니다.

In [ ]:
import os

model_card = """---
language: ko
license: apache-2.0
tags:
  - lora
  - qwen2.5
  - fine-tuning
  - unsloth
base_model: unsloth/Qwen2.5-1.5B-Instruct
---

# Joseon Style Chatbot — LoRA Adapter

Qwen2.5-1.5B-Instruct를 조선시대 말투로 대화하도록 QLoRA로 fine-tuning한 어댑터입니다.

## 모델 정보

| 항목 | 내용 |
|------|------|
| 베이스 모델 | `unsloth/Qwen2.5-1.5B-Instruct` |
| 학습 방법 | QLoRA (4bit, Unsloth) |
| LoRA r | 16 |
| LoRA alpha | 16 |
| 학습 환경 | Google Colab (T4 GPU) |

## 사용법

```python
from unsloth import FastLanguageModel
from peft import PeftModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)

model = PeftModel.from_pretrained(base_model, "YOUR_REPO_ID")
```
"""

with open(os.path.join(LORA_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(model_card)

print("✅ 모델 카드(README.md) 생성 완료")
print(f"   → {LORA_DIR}/README.md")

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "my-username/joseon-qwen2.5-1.5b-lora"  # ← 본인 사용자명으로 변경!

api = HfApi()

# 레포지토리 생성 (이미 있으면 무시)
api.create_repo(REPO_ID, private=False, exist_ok=True)

# 폴더 통째로 업로드
api.upload_folder(
    folder_path=LORA_DIR,
    repo_id=REPO_ID,
)

print(f"✅ 업로드 완료! → https://huggingface.co/{REPO_ID}")

## 5. 업로드된 모델 다운로드 & 테스트

HuggingFace에서 방금 업로드한 LoRA 가중치를 다시 받아와서 정상 동작하는지 확인합니다.

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# 1) 베이스 모델 로드
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True,
)

# 2) HuggingFace에서 LoRA 가중치 다운로드 & 적용
model = PeftModel.from_pretrained(base_model, REPO_ID)

print("✅ HuggingFace에서 모델 로드 완료!")

In [ ]:
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 피곤해.",
    "주말에 어디 가면 좋을까?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

print("\n✅ HuggingFace에서 다운로드한 모델이 정상 동작합니다!")